In [25]:
from pathlib import Path
import pandas as pd
import folium
import osmnx as ox
import numpy as np
import networkx as nx
import osmnx as ox
import os
import json
from pulp import (
    LpProblem, LpMinimize, LpVariable, lpSum, LpBinary,
    LpStatus, value, PULP_CBC_CMD
)

In [3]:
#folder do zapisów csv
OUT_DIR = Path("wroclaw_map_output")
OUT_DIR.mkdir(exist_ok=True)

#folder do zapisów map
OUT_RES_DIR = Path("results")
OUT_RES_DIR.mkdir(exist_ok=True)

### 1. Map of wroclaw, points of laboratories and hospital

In [12]:
###adresy labów

addresses = [
    "pl. Grunwaldzki 18-20, Wroclaw, Poland",
    "Plac Hirszfelda 16/17, Wroclaw, Poland",
    "Św. Macieja 8, Wroclaw, Poland",
    "Bierutowska 17, Wroclaw, Poland",
    "Bonczyka 20, Wroclaw, Poland",
    "Budziszyńska 62a, Wroclaw, Poland",
    "Buforowa 75, Wroclaw, Poland",
    "Canaletta 4, Wroclaw, Poland",
    "Chorwacka 41c, Wroclaw, Poland",
    "Dobrzyńska 21/23, Wroclaw, Poland",
    "Dokerska 2a, Wroclaw, Poland",
    "Gwarna 6a, Wroclaw, Poland",
    "Horbaczewskiego 35, Wroclaw, Poland",
    "Ibn Siny Awicenny 53, Wroclaw, Poland",
    "Jaracza 75h, Wroclaw, Poland",
    "Kasprowicza 9a, Wroclaw, Poland",
    "Kiełczowska 70, Wroclaw, Poland",
    "Krzycka 94, Wroclaw, Poland",
    "Mińska 5, Wroclaw, Poland",
    "Młodych Techników 7, Wroclaw, Poland",
    "Olszewskiego 21, Wroclaw, Poland",
    "Opolska 131, Wroclaw, Poland",
    "Ostrowskiego 3, Wroclaw, Poland",
    "Eluarda 7, Wroclaw, Poland",
    "Pereca 20/1a, Wroclaw, Poland",
    "Piłsudskiego 4a, Wroclaw, Poland",
    "Popowicka 67, Wroclaw, Poland",
    "Powstańców Śląskich 168, Wroclaw, Poland",
    "Powstańców Śląskich 60, Wroclaw, Poland",
    "Rajska 71, Wroclaw, Poland",
    "Sienkiewicza 110, Wroclaw, Poland",
    "Strachocińska 159, Wroclaw, Poland",
    "Swojczycka 69, Wroclaw, Poland",
    "Traugutta 142, Wroclaw, Poland",
    "Trawowa 73, Wroclaw, Poland",
    "Warszawska 2, Wroclaw, Poland",
    "Weigla 12, Wroclaw, Poland",
    "Zakładowa 11h, Wroclaw, Poland",
    "Żelazna 34, Wroclaw, Poland",
    "Zwycięska 41, Wroclaw, Poland"
]


###adres szpitala
hospital_address = "Borowska 213, Wroclaw, Poland"

###zapis punktów
points = []

for addr in addresses:
    lat, lon = ox.geocode(addr)
    points.append({
        "name": addr,
        "lat": lat,
        "lon": lon,
        "type": "lab"
    })

lat, lon = ox.geocode(hospital_address)
points.append({
        "name": hospital_address,
        "lat": lat,
        "lon": lon,
        "type": "hospital"
    })



df = pd.DataFrame(points)
df.to_csv(OUT_DIR / "points_manual.csv", index=False, encoding="utf-8-sig")



# ============================================================
# 3. POBRANIE SIECI DRÓG WROCŁAWIA
#    network_type="drive" = drogi przejezdne dla samochodów
# ============================================================

print("Pobieram sieć drogową Wrocławia...")
G = ox.graph_from_place("Wrocław, Poland", network_type="drive")

#zapis grafu
graph_file = OUT_DIR / "wroclaw_drive.graphml"
ox.save_graphml(G, graph_file)


# Zamiana grafu na GeoDataFrame
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

print("Liczba węzłów:", len(nodes_gdf))
print("Liczba odcinków dróg:", len(edges_gdf))


# ============================================================
# 4. MAPA
# ============================================================

center_lat = df["lat"].mean()
center_lon = df["lon"].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles="OpenStreetMap"
)


# ============================================================
# 5. RYSOWANIE DRÓG
# ============================================================

for _, edge in edges_gdf.iterrows():
    geom = edge.geometry
    if geom is None:
        continue

    coords = [(y, x) for x, y in geom.coords]

    folium.PolyLine(
        locations=coords,
        weight=1,
        opacity=0.25
    ).add_to(m)


# ============================================================
# 6. RYSOWANIE PUNKTÓW
# ============================================================

for idx, row in df.iterrows():
    if row["type"] == "hospital":
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=9,
            popup=f"<b>{row['name']}</b>",
            tooltip=row["name"],
            color="red",
            fill=True,
            fill_opacity=1.0
        ).add_to(m)
    else:
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=6,
            popup=f"{row['name']}",
            tooltip=row["name"],
            color="blue",
            fill=True,
            fill_opacity=0.9
        ).add_to(m)

        # numer obok punktu
        folium.Marker(
            [row["lat"], row["lon"]],
            icon=folium.DivIcon(
                html=f"""
                <div style="font-size: 10pt; color: black; font-weight: bold;">
                    {idx}
                </div>
                """
            )
        ).add_to(m)


# ============================================================
# 7. ZAPIS
# ============================================================

map_path = OUT_DIR / "wroclaw_roads_points.html"
m.save(map_path)

print("Gotowe.")
print("Mapa:", map_path.resolve())
print("Punkty:", (OUT_DIR / "points_manual.csv").resolve())

Pobieram sieć drogową Wrocławia...
Liczba węzłów: 7297
Liczba odcinków dróg: 17220
Gotowe.
Mapa: C:\Users\annad\Desktop\operations\wroclaw_map_output\wroclaw_roads_points.html
Punkty: C:\Users\annad\Desktop\operations\wroclaw_map_output\points_manual.csv


### pkty szpital i droga z jednego lab do szpitala

In [15]:
hospital = df[df["type"] == "hospital"].iloc[0]
lab = df[df["type"] == "lab"].iloc[0]

hospital_node = ox.distance.nearest_nodes(G, hospital["lon"], hospital["lat"])
lab_node = ox.distance.nearest_nodes(G, lab["lon"], lab["lat"])

route = nx.shortest_path(G, hospital_node, lab_node, weight="length")
route_length = nx.shortest_path_length(G, hospital_node, lab_node, weight="length")

route_coords = [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in route]

print(f"Długość trasy: {route_length/1000:.2f} km")

# zapis trasy
route_data = {
    "hospital_name": hospital["name"],
    "lab_name": lab["name"],
    "hospital_node": hospital_node,
    "lab_node": lab_node,
    "route": route,
    "route_length": route_length,
    "route_coords": route_coords
}

with open(os.path.join(OUT_DIR, "route.json"), "w") as f:
    json.dump(route_data, f, indent=2)

print("Zapisano dane trasy: route.json")

# ============================================================
# 5. MAPA TRASY
# ============================================================

m_route = folium.Map(
    location=[df["lat"].mean(), df["lon"].mean()],
    zoom_start=12
)

for _, row in df.iterrows():
    color = "red" if row["type"] == "hospital" else "blue"
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=7,
        color=color,
        fill=True
    ).add_to(m_route)

folium.PolyLine(
    locations=route_coords,
    weight=5,
    opacity=0.9,
    tooltip="Route hospital → lab"
).add_to(m_route)

route_map_path = os.path.join(OUT_DIR, "route_example.html")
m_route.save(route_map_path)

print("Zapisano mapę trasy:", route_map_path)

# ============================================================
# 6. MAPA DRÓG + PUNKTÓW
# ============================================================

m = folium.Map(
    location=[df["lat"].mean(), df["lon"].mean()],
    zoom_start=11,
    tiles="OpenStreetMap"
)

for _, edge in edges_gdf.iterrows():
    geom = edge.geometry
    if geom is None:
        continue
    coords = [(y, x) for x, y in geom.coords]
    folium.PolyLine(locations=coords, weight=1, opacity=0.25).add_to(m)

for idx, row in df.iterrows():
    color = "red" if row["type"] == "hospital" else "blue"
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=8 if row["type"] == "hospital" else 6,
        color=color,
        fill=True
    ).add_to(m)

roads_map_path = os.path.join(OUT_DIR, "wroclaw_roads_points.html")
m.save(roads_map_path)

print("Zapisano mapę dróg:", roads_map_path)
print("Gotowe.")

Długość trasy: 5.97 km
Zapisano dane trasy: route.json
Zapisano mapę trasy: wroclaw_map_output\route_example.html
Zapisano mapę dróg: wroclaw_map_output\wroclaw_roads_points.html
Gotowe.


### stworzenie df nodes

In [19]:
nodes = []
node_id = 0

for p in points:
    if p["type"] == "hospital":
        nodes.append({
            "node_id": 0,
            "name": "Central Hospital",
            "address": p["name"],
            "type": "hospital",
            "lat": p["lat"],
            "lon": p["lon"]
        })
    else:
        node_id += 1
        nodes.append({
            "node_id": node_id,
            "name": f"Lab {node_id}",
            "address": p["name"],
            "type": "lab",
            "lat": p["lat"],
            "lon": p["lon"]
        })

nodes_df = pd.DataFrame(nodes).sort_values("node_id").reset_index(drop=True)
labs_df = nodes_df[nodes_df["type"] == "lab"].reset_index(drop=True)

nodes_df.to_csv(f"{OUT_DIR}/nodes.csv", index=False, encoding="utf-8-sig")
labs_df.to_csv(f"{OUT_DIR}/labs.csv", index=False, encoding="utf-8-sig")

### najbliższy węzeł w grafie dla nodes i zapis do csv

In [22]:
print("Pobieram graf drogowy...")
G = ox.graph_from_place("Wrocław, Poland", network_type="drive")

# Dodanie prędkości i czasu przejazdu do krawędzi
G = ox.routing.add_edge_speeds(G)
G = ox.routing.add_edge_travel_times(G)

# ============================================================
# 4. PRZYPISANIE PUNKTÓW DO NAJBLIŻSZYCH WĘZŁÓW GRAFU
# ============================================================

# Uwaga: X = longitude, Y = latitude
nearest_nodes = ox.distance.nearest_nodes(
    G,
    X=nodes_df["lon"].values,
    Y=nodes_df["lat"].values
)

nodes_df["osmnx_node"] = nearest_nodes

nodes_df.to_csv(OUT_DIR / "nodes_with_osm_ids.csv", index=False, encoding="utf-8-sig")
print("Zapisano nodes_with_osm_ids.csv")

Pobieram graf drogowy...
Zapisano nodes_with_osm_ids.csv


### macierz czasów przejazdu

In [23]:
# ============================================================
# 5. MACIERZ CZASÓW PRZEJAZDU
# ============================================================

node_ids = nodes_df["node_id"].tolist()
names = nodes_df["name"].tolist()
osm_nodes = nodes_df["osmnx_node"].tolist()

n = len(nodes_df)

time_matrix = pd.DataFrame(
    data=np.full((n, n), np.nan),
    index=node_ids,
    columns=node_ids
)

distance_matrix = pd.DataFrame(
    data=np.full((n, n), np.nan),
    index=node_ids,
    columns=node_ids
)

print("Liczę macierz czasów przejazdu...")

for i in range(n):
    source_osm = osm_nodes[i]

    # najkrótsze czasy od jednego źródła do wszystkich
    travel_times = nx.single_source_dijkstra_path_length(
        G,
        source_osm,
        weight="travel_time"
    )

    distances = nx.single_source_dijkstra_path_length(
        G,
        source_osm,
        weight="length"
    )

    for j in range(n):
        target_osm = osm_nodes[j]

        if target_osm in travel_times:
            # travel_time w sekundach -> minuty
            time_matrix.iloc[i, j] = travel_times[target_osm] / 60.0

        if target_osm in distances:
            # length w metrach
            distance_matrix.iloc[i, j] = distances[target_osm]

# ============================================================
# 6. ZAPIS MACIERZY
# ============================================================

time_matrix.to_csv(OUT_DIR / "travel_time_matrix_min.csv", encoding="utf-8-sig")
distance_matrix.to_csv(OUT_DIR / "distance_matrix_m.csv", encoding="utf-8-sig")

print("Zapisano:")
print(OUT_DIR / "travel_time_matrix_min.csv")
print(OUT_DIR / "distance_matrix_m.csv")

Liczę macierz czasów przejazdu...
Zapisano:
wroclaw_map_output\travel_time_matrix_min.csv
wroclaw_map_output\distance_matrix_m.csv


### wersja long format - czasy

In [24]:
rows = []
for i in range(n):
    for j in range(n):
        rows.append({
            "from_node_id": node_ids[i],
            "from_name": names[i],
            "to_node_id": node_ids[j],
            "to_name": names[j],
            "travel_time_min": time_matrix.iloc[i, j],
            "distance_m": distance_matrix.iloc[i, j],
        })

travel_long_df = pd.DataFrame(rows)
travel_long_df.to_csv(OUT_DIR / "travel_times_long.csv", index=False, encoding="utf-8-sig")

print("Zapisano także:")
print(OUT_DIR / "travel_times_long.csv")
print("Gotowe.")

Zapisano także:
wroclaw_map_output\travel_times_long.csv
Gotowe.
